# 04. 피처 엔지니어링 및 최종 XGBoost

기본 10개 행동 변수에 상호작용·비율·플래그·순환형·로그 피처를 추가해
**42개 피처**를 구성합니다.

최종 운영 기준은 Recall을 우선하여 **Threshold 0.35**를 사용합니다.

이 노트북을 실행하면 `models/`에 Streamlit/FastAPI가 사용하는 모델 자산이 생성됩니다.

In [ ]:
from pathlib import Path

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    if current.name == "notebooks":
        return current.parent
    if (current / "notebooks").exists():
        return current
    return current.parent

PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SAMPLE_DIR = PROJECT_ROOT / "data" / "sample"
MODEL_DIR = PROJECT_ROOT / "models"

for directory in [INTERIM_DIR, PROCESSED_DIR, SAMPLE_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score,
)
from xgboost import XGBClassifier

master = pd.read_csv(
    PROCESSED_DIR / "model_experiment_data.csv",
    parse_dates=["trial_date"],
)

print("master:", master.shape)

In [ ]:
EPS = 1e-6

BASE_FEATURES = [
    "avg_stay_hour", "avg_daily_enter", "visit_days", "first_visit_delay",
    "consecutive_group_2일", "consecutive_group_3일", "first_visit_hour",
    "n_sites_visited", "area_pyeong", "is_post_covid",
]

def add_features(df):
    df = df.copy()

    df["trial_month"] = df["trial_date"].dt.month
    df["trial_dayofweek"] = df["trial_date"].dt.dayofweek
    df["is_weekend_trial"] = (df["trial_dayofweek"] >= 5).astype(int)

    df["stay_x_visit"] = df["avg_stay_hour"] * df["visit_days"]
    df["enter_x_visit"] = df["avg_daily_enter"] * df["visit_days"]
    df["stay_x_enter"] = df["avg_stay_hour"] * df["avg_daily_enter"]

    df["enter_per_visit_day"] = df["avg_daily_enter"] / (df["visit_days"] + EPS)
    df["stay_per_enter"] = df["avg_stay_hour"] / (df["avg_daily_enter"] + EPS)
    df["delay_per_visit_day"] = df["first_visit_delay"] / (df["visit_days"] + EPS)
    df["visit_delay_interaction"] = df["visit_days"] * df["first_visit_delay"]

    df["is_fast_visit"] = (df["first_visit_delay"] <= 1).astype(int)
    df["is_delayed_visit"] = (df["first_visit_delay"] >= 3).astype(int)
    df["is_frequent_user"] = (df["avg_daily_enter"] >= 2).astype(int)
    df["is_long_stay"] = (df["avg_stay_hour"] >= 2).astype(int)
    df["is_short_frequent"] = (
        (df["avg_stay_hour"] < 1) & (df["avg_daily_enter"] >= 2)
    ).astype(int)
    df["is_long_frequent"] = (
        (df["avg_stay_hour"] >= 2) & (df["avg_daily_enter"] >= 2)
    ).astype(int)

    df["is_morning"] = df["first_visit_hour"].between(6, 11).astype(int)
    df["is_afternoon"] = df["first_visit_hour"].between(12, 17).astype(int)
    df["is_evening"] = df["first_visit_hour"].between(18, 23).astype(int)

    df["first_visit_hour_sin"] = np.sin(2 * np.pi * df["first_visit_hour"] / 24)
    df["first_visit_hour_cos"] = np.cos(2 * np.pi * df["first_visit_hour"] / 24)
    df["trial_month_sin"] = np.sin(2 * np.pi * df["trial_month"] / 12)
    df["trial_month_cos"] = np.cos(2 * np.pi * df["trial_month"] / 12)

    df["post_covid_x_visit_days"] = df["is_post_covid"] * df["visit_days"]
    df["post_covid_x_avg_stay"] = df["is_post_covid"] * df["avg_stay_hour"]
    df["post_covid_x_daily_enter"] = df["is_post_covid"] * df["avg_daily_enter"]

    for col in [
        "avg_stay_hour", "avg_daily_enter", "first_visit_delay",
        "area_pyeong", "n_sites_visited", "stay_x_visit",
    ]:
        df[f"log1p_{col}"] = np.log1p(df[col].clip(lower=0))

    return df

master_fe = add_features(master)

In [ ]:
FEATURE_COLS = BASE_FEATURES + [
    "stay_x_visit",
    "enter_x_visit",
    "stay_x_enter",
    "enter_per_visit_day",
    "stay_per_enter",
    "delay_per_visit_day",
    "visit_delay_interaction",
    "is_fast_visit",
    "is_delayed_visit",
    "is_frequent_user",
    "is_long_stay",
    "is_short_frequent",
    "is_long_frequent",
    "is_morning",
    "is_afternoon",
    "is_evening",
    "first_visit_hour_sin",
    "first_visit_hour_cos",
    "trial_month",
    "trial_dayofweek",
    "is_weekend_trial",
    "trial_month_sin",
    "trial_month_cos",
    "post_covid_x_visit_days",
    "post_covid_x_avg_stay",
    "post_covid_x_daily_enter",
    "log1p_avg_stay_hour",
    "log1p_avg_daily_enter",
    "log1p_first_visit_delay",
    "log1p_area_pyeong",
    "log1p_n_sites_visited",
    "log1p_stay_x_visit",
]

assert len(FEATURE_COLS) == 42

X = master_fe[FEATURE_COLS].copy()
y = master_fe["is_payment"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

feature_medians = X_train.median(numeric_only=True)
X_train = X_train.fillna(feature_medians)
X_test = X_test.fillna(feature_medians)

print("X shape:", X.shape)
print("피처 수:", len(FEATURE_COLS))

In [ ]:
tuned_model = XGBClassifier(
    n_estimators=700,
    learning_rate=0.02,
    max_depth=2,
    gamma=0.3,
    colsample_bytree=0.7,
    scale_pos_weight=2.0,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)

tuned_model.fit(X_train, y_train)

THRESHOLD = 0.35
proba = tuned_model.predict_proba(X_test)[:, 1]
pred = (proba >= THRESHOLD).astype(int)

metrics = {
    "accuracy": float(accuracy_score(y_test, pred)),
    "precision": float(precision_score(y_test, pred)),
    "recall": float(recall_score(y_test, pred)),
    "f1_score": float(f1_score(y_test, pred)),
    "roc_auc": float(roc_auc_score(y_test, proba)),
    "threshold": THRESHOLD,
    "feature_count": len(FEATURE_COLS),
}

pd.Series(metrics).round(4)

> 원본 데이터 버전 및 전처리 세부 순서에 따라 재학습 지표는 기존 시연 모델과 소폭 달라질 수 있습니다.
> 저장되는 피처 구성·하이퍼파라미터·Threshold는 현재 Streamlit/FastAPI 구조와 동일하게 맞춥니다.

In [ ]:
# 모델 배포 자산 저장
joblib.dump(tuned_model, MODEL_DIR / "best_tuned_model.pkl")
joblib.dump(THRESHOLD, MODEL_DIR / "best_tuned_threshold.pkl")

with (MODEL_DIR / "feature_cols.json").open("w", encoding="utf-8") as f:
    json.dump(FEATURE_COLS, f, ensure_ascii=False, indent=2)

with (MODEL_DIR / "feature_medians.json").open("w", encoding="utf-8") as f:
    json.dump(
        {k: float(v) for k, v in feature_medians.to_dict().items()},
        f,
        ensure_ascii=False,
        indent=2,
    )

with (MODEL_DIR / "model_metrics.json").open("w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print("생성 파일")
for name in [
    "best_tuned_model.pkl",
    "best_tuned_threshold.pkl",
    "feature_cols.json",
    "feature_medians.json",
    "model_metrics.json",
]:
    print("-", MODEL_DIR / name)